# M5 경제표현 + 양성 구매금액 가중학습: Dunnhumby test-only 10 seeds

검증 구간을 만들지 않습니다. DAY 1~697을 모두 학습하고, 고정 100 epoch의 마지막 checkpoint를 DAY 698~704 test에서 seed별 한 번만 평가합니다. DAY 705~711은 사용하지 않습니다. 이미 완료된 seed 42의 A~F 결과는 재학습·재평가하지 않고 검증 후 재사용하며, 이 노트북에서는 seed 43~51만 새로 학습합니다. 수식·rho·lambda·epoch·판정기준은 seed 42와 동일하게 고정합니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import importlib, os, shutil, subprocess, sys

REVIEWED_SHA = '28cf7b2ab8ad0f9e05a1ff8f0af02da378da845d'
REPO_URL = 'https://github.com/jung-un/clv-m2-lightgcn-runner.git'
os.chdir('/content')
repo = Path('/content/clv-m2-lightgcn-runner')
clone_errors = []
for clone_attempt in range(1, 4):
    if repo.exists():
        shutil.rmtree(repo)
    result = subprocess.run(
        ['git', 'clone', REPO_URL, str(repo)],
        text=True, capture_output=True,
    )
    if result.returncode == 0:
        break
    clone_errors.append(result.stderr.strip())
    print(f'GitHub clone {clone_attempt}/3 실패:', result.stderr.strip())
else:
    raise RuntimeError('GitHub clone 3회 실패:\n' + '\n'.join(clone_errors))
subprocess.run(['git', '-C', str(repo), 'checkout', '-q', REVIEWED_SHA], check=True)
actual_sha = subprocess.check_output(
    ['git', '-C', str(repo), 'rev-parse', 'HEAD'], text=True
).strip()
assert actual_sha == REVIEWED_SHA, (actual_sha, REVIEWED_SHA)
for module_name in tuple(sys.modules):
    if module_name.startswith(('lightgcn_', 'clv_')):
        del sys.modules[module_name]
importlib.invalidate_caches()
%cd /content/clv-m2-lightgcn-runner
print('실행 코드 고정 완료:', actual_sha)

In [ ]:
import json
import torch
from lightgcn_clv_m5_economic_positive_weight_test import (
    FULL_SEEDS,
    configure_m5_economic_positive_test_run,
    preflight_summary,
    run_m5_economic_positive_test,
)

assert torch.cuda.is_available(), '런타임 유형에서 GPU를 선택하세요.'
PILOT_RESULT = '/content/drive/MyDrive/논문/data/results_v3_dunnhumby_m5_economic_positive_weighting_test_seed42_v1/m5_economic_positive_weight_test_b22a507c8ab5.json'
assert Path(PILOT_RESULT).is_file(), f'seed 42 결과가 없습니다: {PILOT_RESULT}'
cfg = configure_m5_economic_positive_test_run(
    seeds=FULL_SEEDS,
    reused_seed42_json=PILOT_RESULT,
    out_dir='/content/drive/MyDrive/논문/data/results_v3_dunnhumby_m5_economic_positive_weighting_test_multiseed_v1',
)
summary = preflight_summary(cfg)
assert cfg.seeds == tuple(range(42, 52))
assert summary['seed42_handling'] == 'reuse the completed seed-42 test result; train seeds 43--51 only'
assert summary['validation_constructed'] is False
assert summary['holdout_evaluation'] is False
print(json.dumps(summary, ensure_ascii=False, indent=2))

In [ ]:
result_df = run_m5_economic_positive_test(cfg)

In [ ]:
from IPython.display import display

def show(frame):
    view = frame.copy()
    view.attrs = {}
    display(view)

print('1) seed 42~51 test 절대지표')
show(result_df)
print('2) 10시드 모델별 평균·표준편차·95% t 구간')
show(result_df.attrs['mean'])
print('3) 동일 seed 대조군 비교')
show(result_df.attrs['comparison'])
print('4) 동일 seed 대응차 평균·표준편차·95% t 구간·양수 seed 수')
show(result_df.attrs['paired_mean'])
print("5) seed별 M2 × M4' 상호작용")
show(result_df.attrs['interaction'])
print('6) seed별 사전 고정 기준 판독')
print(json.dumps(result_df.attrs['descriptive_reading'], ensure_ascii=False, indent=2))
print('7) 저장 파일')
print(json.dumps(result_df.attrs['result_paths'], ensure_ascii=False, indent=2))